# deep-learning_001

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (43).ipynb`

| Field | Value |
|---|---|
| Section | `deep_learning` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 13 |
| Detected functions | _cosine_similarity, _adjacent_similarities, _normalize, _inverse_entropy, compute_signals, generate_insights, analyze_video |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


# 🧠 NeuroAD Video Analysis Pipeline

**Run your video through TRIBE v2 brain encoding model on Google Colab T4 GPU**

This notebook:
1. Sets up the TRIBE v2 model from Meta/Facebook Research
2. Processes your video to predict brain responses (VIDEO-ONLY for speed)
3. Returns 12 cognitive signals (0-10 scale) with actionable insights

---
⚠️ **Before running**: Go to `Runtime → Change runtime type → T4 GPU`

## Step 0: Mount Google Drive (Recommended)

This saves model weights to your Drive so you don't re-download them (~6GB) each session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

## Step 1: Verify GPU

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

## Step 2: Install Dependencies

In [ ]:
%%capture
# Force numpy to compatible version FIRST
!pip uninstall -y numpy
!pip install -q "numpy==1.26.4"

# Core packages
!pip install -q fastapi uvicorn python-multipart
!pip install -q opencv-python moviepy
!pip install -q pandas scikit-learn

# Don't upgrade torch - use Colab's pre-installed version
!pip install -q transformers huggingface-hub

# TRIBE v2 dependencies
!pip install -q einops pyyaml x-transformers
!pip install -q spacy gtts langdetect soundfile Levenshtein julius
!pip install -q pydantic tqdm requests aiofiles

print("✅ All dependencies installed!")

In [ ]:
# Verify installations
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("\n✅ Core packages verified!")

## Step 3: Clone TRIBE v2 Repository

In [ ]:
import os
from pathlib import Path

# Setup directories (use Google Drive for persistence)
BASE_DIR = Path("/content/drive/MyDrive/")
TRIBE_SRC = BASE_DIR / "tribev2-src"
CACHE_DIR = BASE_DIR / "cache"

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Clone TRIBE v2 repository
if not TRIBE_SRC.exists():
    print("📥 Cloning TRIBE v2 repository...")
    !git clone --depth 1 https://github.com/facebookresearch/tribev2.git {TRIBE_SRC}
    print("✅ TRIBE v2 repository cloned!")
else:
    print("✅ TRIBE v2 repository already exists")

# Install TRIBE v2 package
print("\n📦 Installing TRIBE v2 package...")
!pip install -q --no-deps -e {TRIBE_SRC}
print("✅ TRIBE v2 package installed!")

## Step 4: HuggingFace Authentication

TRIBE v2 uses gated models requiring approval:
1. Get a token from https://huggingface.co/settings/tokens
2. Request access to:
   - https://huggingface.co/meta-llama/Llama-3.2-3B
   - https://huggingface.co/facebook/tribev2

In [ ]:
from huggingface_hub import login

# Login to HuggingFace (paste your token when prompted)
login()

## Step 5: Load TRIBE v2 Model (VIDEO-ONLY MODE)

⚡ This configures TRIBE to process **video only** (no audio, no text) for 10x faster analysis.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
from tribev2 import TribeModel

print("🧠 Loading TRIBE v2 model in VIDEO-ONLY mode...")
print(f"   Cache directory: {CACHE_DIR}")
print("   ⚠️ First download: ~4GB, takes 3-5 minutes")
print("   ⚡ Subsequent runs: <30 seconds (cached to Drive)\n")

# Configure for GPU + VIDEO-ONLY
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"   Device: {device}")

# CRITICAL: Configure for VIDEO-ONLY processing
config_update = {
    "data.video_feature.image.device": device,
    # "data.modalities": ["video"],  # Removed this line as it caused the error
}

model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=str(CACHE_DIR),
    device=device,
    config_update=config_update
)

print("\n✅ TRIBE v2 model loaded successfully!")
print("✅ Configured for VIDEO-ONLY (fast mode)")

## Step 6: Define Analysis Functions

In [ ]:
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple

# ============ Signal Mapping Functions ============

def _cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

def _adjacent_similarities(matrix: np.ndarray) -> np.ndarray:
    if matrix.shape[0] < 2:
        return np.array([0.0], dtype=float)
    return np.array([
        _cosine_similarity(matrix[i], matrix[i + 1])
        for i in range(matrix.shape[0] - 1)
    ])

def _normalize(value: float, low: float, high: float) -> float:
    clipped = min(max(value, low), high)
    scaled = (clipped - low) / (high - low)
    return round(10.0 * scaled, 2)

def _inverse_entropy(values: np.ndarray, bins: int = 32) -> float:
    hist, _ = np.histogram(values, bins=bins, density=True)
    p = hist / (hist.sum() + 1e-12)
    entropy = -np.sum(p * np.log(p + 1e-12))
    max_entropy = np.log(bins)
    return float(1.0 - (entropy / max_entropy))

def compute_signals(brain_vectors: List[np.ndarray]) -> Dict[str, float]:
    """Convert brain vectors to 12 cognitive signals (0-10 scale)"""
    if not brain_vectors:
        raise ValueError("No brain vectors provided")

    matrix = np.vstack(brain_vectors)
    if matrix.shape[1] < 800:
        raise ValueError(f"Brain vectors must have at least 800 dimensions, got {matrix.shape[1]}")

    adjacent_sim = _adjacent_similarities(matrix)
    repetition_raw = float(np.mean(adjacent_sim))
    novelty_raw = float(1.0 - repetition_raw)
    temporal_drift = float(np.mean(np.linalg.norm(np.diff(matrix, axis=0), axis=1))) if len(matrix) > 1 else 0.0
    coherence_raw = float(1.0 / (1.0 + temporal_drift))

    raw = {
        "saliency": float(np.mean(matrix[:, 0:100])),
        "motion": float(np.std(matrix[:, 100:200])),
        "novelty": novelty_raw,
        "emotion_intensity": float(np.mean(matrix[:, 200:300])),
        "emotion_valence": float(np.mean(matrix[:, 300:350])),
        "relatability": float(np.mean(matrix[:, 350:400])),
        "distinctiveness": float(np.var(matrix[:, 400:500])),
        "repetition": repetition_raw,
        "story_coherence": coherence_raw,
        "cognitive_load": float(np.mean(matrix[:, 500:650])),
        "clarity": _inverse_entropy(matrix[:, 0:800].ravel()),
        "info_density": float(np.mean(matrix[:, 650:800])),
    }

    return {
        "saliency": _normalize(raw["saliency"], -3.0, 3.0),
        "motion": _normalize(raw["motion"], 0.0, 3.0),
        "novelty": _normalize(raw["novelty"], -1.0, 1.5),
        "emotion_intensity": _normalize(raw["emotion_intensity"], -3.0, 3.0),
        "emotion_valence": _normalize(raw["emotion_valence"], -3.0, 3.0),
        "relatability": _normalize(raw["relatability"], -3.0, 3.0),
        "distinctiveness": _normalize(raw["distinctiveness"], 0.0, 5.0),
        "repetition": _normalize(raw["repetition"], -1.0, 1.0),
        "story_coherence": _normalize(raw["story_coherence"], 0.0, 1.0),
        "cognitive_load": _normalize(raw["cognitive_load"], -3.0, 3.0),
        "clarity": _normalize(raw["clarity"], 0.0, 1.0),
        "info_density": _normalize(raw["info_density"], -3.0, 3.0),
    }

# ============ Insight Generation ============

SIGNAL_RULES: Dict[str, Tuple[float, float, str, str]] = {
    "saliency": (4.5, 8.5, "Weak visual attention → improve contrast, framing, or movement in key scenes.", "Excellent visual saliency → maintain current visual impact strategy."),
    "motion": (4.0, 8.0, "Low motion signal → add dynamic camera cuts or object movement for attention retention.", "Strong motion dynamics → ensure motion doesn't overwhelm the core message."),
    "novelty": (4.5, 8.0, "Low novelty → content feels predictable; introduce surprising visual or narrative elements.", "High novelty → ensure audience can still follow the core message."),
    "emotion_intensity": (4.5, 8.5, "Low emotional intensity → introduce stronger emotional hooks in opening seconds.", "Strong emotional engagement → leverage this for brand recall moments."),
    "emotion_valence": (3.5, 7.5, "Negative emotional tone → rebalance with more positive or reassuring cues.", "Positive emotional tone → amplify feel-good moments for sharing potential."),
    "relatability": (4.5, 8.0, "Low relatability → add recognizable human context and audience-aligned scenarios.", "High relatability → use for testimonial or UGC-style content extensions."),
    "distinctiveness": (4.5, 8.0, "Low distinctiveness → creative appears generic; introduce a unique visual motif.", "High distinctiveness → ensure brand elements are consistently present."),
    "story_coherence": (5.0, 8.5, "Weak story coherence → tighten narrative sequence and improve transition logic.", "Strong narrative flow → consider longer-form content to leverage storytelling."),
    "cognitive_load": (3.0, 7.0, "Very low cognitive load → add complexity or depth to maintain engagement.", "High cognitive load → simplify concurrent visual elements and reduce simultaneous claims."),
    "clarity": (5.0, 8.5, "Low clarity → sharpen value proposition and make the core message explicit earlier.", "Excellent clarity → use as template for future campaigns."),
    "info_density": (3.5, 7.0, "Low information density → add more value-packed content per second.", "High information density → reduce clutter and prioritize one key takeaway per scene."),
    "repetition": (4.0, 7.0, "Low repetition → key elements aren't reinforced; increase visual/audio repetition of brand.", "High repetition → add variation to prevent audience fatigue."),
}

def generate_insights(signals: Dict[str, float]) -> List[str]:
    """Generate actionable insights based on cognitive signal values"""
    insights: List[str] = []
    strengths: List[str] = []

    for signal, (low_thresh, high_thresh, low_insight, high_insight) in SIGNAL_RULES.items():
        value = signals.get(signal, 5.0)
        if value < low_thresh:
            insights.append(low_insight)
        elif value > high_thresh:
            strengths.append(high_insight)

    if strengths and not insights:
        insights.append(strengths[0])

    if not insights:
        insights.append("Balanced cognitive-emotional profile → iterate with A/B creative variants for incremental lift.")

    avg_score = sum(signals.values()) / len(signals) if signals else 5.0
    if avg_score >= 7.0:
        insights.insert(0, f"⭐ Strong overall performance (score: {avg_score:.1f}/10)")
    elif avg_score <= 4.5:
        insights.insert(0, f"⚠️ Below-average performance (score: {avg_score:.1f}/10) - prioritize improvements")

    return insights

print("✅ Analysis functions defined!")

## Step 7: Main Analysis Function (VIDEO-ONLY)

In [ ]:
import time

def analyze_video(video_path: str) -> Dict:
    """
    Complete video analysis pipeline (VIDEO-ONLY mode):
    1. Run TRIBE v2 to get brain responses
    2. Map to 12 cognitive signals
    3. Generate actionable insights

    Processing time: ~1-3 seconds on T4 GPU
    """
    print(f"\n🎬 Analyzing: {video_path}")
    start_time = time.time()

    # Step 1: Create video-only event (skip audio/text)
    print("🧠 Running TRIBE v2 brain encoding (video-only)...")

    event = {
        "type": "Video",
        "filepath": str(video_path),
        "start": 0,
        "timeline": "default",
        "subject": "default",
    }
    events = pd.DataFrame([event])

    # Run TRIBE v2 inference with mixed precision
    with torch.amp.autocast('cuda'):
        preds, segments = model.predict(events, verbose=False)

    # Convert predictions to brain vectors
    brain_vectors = [np.asarray(p).ravel() for p in preds]
    print(f"   → Got {len(brain_vectors)} brain vectors (dim: {brain_vectors[0].shape[0]})")

    # Step 2: Compute cognitive signals
    print("📊 Computing cognitive signals...")
    signals = compute_signals(brain_vectors)

    # Step 3: Generate insights
    print("💡 Generating insights...")
    insights = generate_insights(signals)

    elapsed = time.time() - start_time
    overall_score = sum(signals.values()) / len(signals)

    print(f"✅ Analysis complete in {elapsed:.1f}s")

    return {
        "signals": signals,
        "insights": insights,
        "overall_score": round(overall_score, 2),
        "timing_seconds": round(elapsed, 2),
        "num_segments": len(brain_vectors)
    }

print("✅ Analysis function ready!")

## Step 8: Upload Your Video

In [ ]:
from google.colab import files

print("📤 Upload your video file (MP4 or MOV):")
uploaded = files.upload()

if uploaded:
    video_filename = list(uploaded.keys())[0]
    video_path = f"/content/{video_filename}"
    print(f"\n✅ Uploaded: {video_filename}")
else:
    print("❌ No file uploaded")

## Step 9: Run Analysis!

In [ ]:
# Run the full analysis pipeline
results = analyze_video(video_path)

# Display results
print("\n" + "="*60)
print("🧠 NEUROADV2 ANALYSIS RESULTS")
print("="*60)

print(f"\n⏱️  Processing time: {results['timing_seconds']}s")
print(f"📊 Overall Score: {results['overall_score']}/10")
print(f"🎞️  Segments analyzed: {results['num_segments']}")

print("\n📈 COGNITIVE SIGNALS (0-10 scale):")
print("-" * 40)
for signal, value in results['signals'].items():
    bar = "█" * int(value) + "░" * (10 - int(value))
    print(f"  {signal:20s} {bar} {value}")

print("\n💡 INSIGHTS:")
print("-" * 40)
for i, insight in enumerate(results['insights'], 1):
    print(f"  {i}. {insight}")

print("\n" + "="*60)

## Step 10: Export Results (Optional)

In [ ]:
import json

# Save results to JSON
output_file = f"/content/{video_filename.rsplit('.', 1)[0]}_analysis.json"
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"📁 Results saved to: {output_file}")

# Download the results
files.download(output_file)
print("✅ Results downloaded!")

---

## ⚡ Performance Notes

**Current pipeline (video-only):**
- ✅ T4 GPU: **1-3 seconds** per video
- ✅ No audio extraction (saves 4+ minutes)
- ✅ No text transcription
- ✅ Visual brain responses only

**Model downloads (first time only):**
- Video model: ~4GB (3-5 minutes)
- Saved to Google Drive for instant future loads

---

## 🔄 Analyze Another Video

Just run the cell below to analyze additional videos:

In [ ]:
# Upload another video
uploaded = files.upload()
if uploaded:
    video_filename = list(uploaded.keys())[0]
    video_path = f"/content/{video_filename}"

    # Run analysis
    results = analyze_video(video_path)

    # Display results
    print(f"\n📊 Overall Score: {results['overall_score']}/10")
    print("\n📈 Signals:")
    for signal, value in results['signals'].items():
        bar = "█" * int(value) + "░" * (10 - int(value))
        print(f"  {signal:20s} {bar} {value}")
    print("\n💡 Insights:")
    for insight in results['insights']:
        print(f"  • {insight}")

---

## 🐛 Troubleshooting

### Model download stuck?
```python
# Runtime → Restart runtime, then re-run from Step 5
# The partially downloaded model is cached and will resume
```

### Want to enable audio/text processing?
```python
# Change Step 5 config_update to:
config_update = {
    "data.video_feature.image.device": device,
    "data.text_feature.device": device,
    "data.audio_feature.device": device,
    "data.modalities": ["video", "audio", "text"],  # Enable all modalities
}
# Then use model.get_events_dataframe() in analyze_video()
# Warning: 10x slower (5-10s vs 1-3s)
```

### CUDA out of memory?
```python
# Runtime → Restart runtime
# Use shorter videos (<60 seconds)
```

---

## ℹ️ Signal Reference

| Category | Signal | Description |
|----------|--------|-------------|
| **Attention** | `saliency` | Visual attention strength |
| | `motion` | Dynamic movement detection |
| | `novelty` | Uniqueness vs. previous content |
| **Emotion** | `emotion_intensity` | Emotional activation level |
| | `emotion_valence` | Positive vs. negative emotion |
| | `relatability` | Audience connection potential |
| **Memory** | `distinctiveness` | Memorability potential |
| | `repetition` | Pattern consistency |
| | `story_coherence` | Narrative flow quality |
| **Cognitive** | `cognitive_load` | Mental processing complexity |
| | `clarity` | Message sharpness |
| | `info_density` | Information concentration |